In [ ]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
        print('{} memory growth: {}'.format(device, tf.config.experimental.get_memory_growth(device)))
else:
    print("Not enough GPU hardware devices available")

In [ ]:
from tensorflow import keras
from keras.models import Model
from keras.utils import np_utils
from PIL import Image
import numpy as np
import glob
import matplotlib.pyplot as plt
import os
import pandas as pd  
import seaborn as sn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from keras.models import model_from_json
import openpyxl

In [ ]:
seed = 0
np.random.seed(seed)

fold_number = 8

In [ ]:
os.makedirs("../performance_metrics/tissue", exist_ok = True)
os.makedirs("../figure/tissue/extest", exist_ok = True)
os.makedirs("../result_dir", exist_ok = True)

In [ ]:
#Place the four image folders, each corresponding to predection labels, into the train folder.

os.makedirs("../data/train", exist_ok = True)
os.makedirs("../data/test", exist_ok = True)

# Place the four test folders into either the "tissue", "culture" or "neuronmixture" folder.
folder_path="../data/test/tissue/"
folder = os.listdir("../data/test/tissue")

# folder_path="../data/test/culture/"
# folder = os.listdir("../data/test/culture")

# folder_path="../data/test/neuronmixture/"
# folder = os.listdir("../data/test/neuronmixture")

predict_label_path=os.listdir("../data/train/tissue_augmentate")
#predict_label_path=os.listdir("../data/train/culture_augmentate")
#predict_label_path=os.listdir("../data/train/neuronmixture_augmentate")

true_label_path=os.listdir("../data/test/tissue")
#true_label_path=os.listdir("../data/test/culture")
#true_label_path=os.listdir("../data/test/neuronmixture")

model_name="tissue"
model_path="../train/CNNmodels/tissue/tissue_model"

# model_name="culture"
# model_path="../train/CNNmodels/culture/culture_model"

# model_name="neuronmixture"
# model_path="../train/CNNmodels/neuronmixture/neuronmixture_model"

In [ ]:
image_size = 150
dense_size  = len(folder)
classes_number = len(folder)

X_extest = []
y_extest = []
for index, name in enumerate(folder):
    dir = folder_path + name
    
    images = glob.glob(dir + "/*.tif")
    for image in images:
        image1 = Image.open(image)
        data = np.asarray(image1, 'float32')
        X_extest.append(data)
        y_extest.append(index)

X_extest = np.array(X_extest)
y_extest = np.array(y_extest)
X_extest = X_extest / 255.0
X_extest = X_extest.reshape(X_extest.shape[0], X_extest.shape[1], X_extest.shape[2], 1)

y_extest = np_utils.to_categorical(y_extest, dense_size)

In [ ]:
for rd in range(1, fold_number+1):
    json_string = open(model_path+str(rd)+".json", "r").read()
    model = model_from_json(json_string)
    model.load_weights(model_path+str(rd)+"_weights.hdf5")
    
    y_pred = np.argmax(model.predict(X_extest), axis=1)
    y_true = np.argmax(y_extest, axis=1)

    labels = sorted(list(set(y_true)|set(y_pred)))
    cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    cmx_data_list = cmx_data.tolist()

    df_cmx = pd.DataFrame(cmx_data, index=true_label_path, columns=predict_label_path)
    fig=plt.figure(figsize = (10,7))
    sn.heatmap(df_cmx, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix-"+str(rd))
    plt.xlabel("predict_classes")
    plt.ylabel("true_classes")
    plt.tight_layout()
    plt.rcParams["font.size"] = 12
    fig.savefig("../figure/"+model_name+"/extest/cmx_test_ensemble_"+str(rd)+".tiff", dpi=600)  
    
    if rd == 1:
        y_pred_all = y_pred
        y_true_all = y_true
    else:
        y_pred_all = np.append(y_pred_all, y_pred)
        y_true_all = np.append(y_true_all, y_true)

In [ ]:
labels = sorted(list(set(y_true_all)|set(y_pred_all)))
cmx_data = confusion_matrix(y_true_all, y_pred_all, labels=labels)
df_cmx = pd.DataFrame(cmx_data, index=true_label_path, columns=predict_label_path)
fig=plt.figure(figsize = (10,7))
sn.heatmap(df_cmx, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("predict_classes")
plt.ylabel("true_classes")
plt.rcParams["font.size"] = 12
plt.tight_layout()
fig.savefig("../figure/"+model_name+"/extest/cmx_test_ensemble_"+str(rd)+"8hold.tiff", dpi=600)  
plt.show()

precision_recall_fscore_support(y_true_all, y_pred_all, average="macro")
print(classification_report(y_true_all, y_pred_all, digits=6))